# Assignment C4: Custom ReAct Agent with Configurable Personas

**Course:** EECE 503P/798S: Agentic Systems  
**Assignment:** Build a Custom ReAct Agent with Configurable Personas  
**Framework:** LangGraph (State Machine-based ReAct Workflow)  
**Use Case:** WanderWise Travel Concierge & Trip Budget Reasoning Agent  

---

## 1. Use Case Summary and Write-Up (10 Points)

### 1.1 Scenario & Business Goal
This project extends **WanderWise Travel** (the fictional boutique travel planning business from Assignment C3) by transforming it into an autonomous **ReAct-style reasoning agent** (*Reasoning + Acting*).  
Instead of generating answers in a single pass, the agent employs a deliberate loop:
$$\text{Thought} \longrightarrow \text{Action} \longrightarrow \text{Observation} \longrightarrow \text{Thought} \longrightarrow \text{Final Answer}$$

* **Agent Goal:** Help prospective travelers discover curated vacation packages, calculate exact multi-traveler budgets with optional upgrades, audit inclusion/exclusion policies, and log qualified leads without human intervention.
* **Target Audience:** Independent vacationers, couples planning honeymoons, and travelers seeking personalized itinerary quotes.
* **Constraints & Operational Boundaries:**
  1. All packages ($1,850 Bali, $1,650 Rome, $1,950 Santorini, $2,150 Swiss Alps, $2,800 Japan) are fictional.
  2. International airfare and visas are strictly excluded from all packages; the agent must never claim it booked flights or issued visas.
  3. Any unapproved inquiries (e.g. cancellation terms, unannounced tours) must be logged via tools rather than speculated.

### 1.2 Available ReAct Tools
The agent is equipped with five distinct, single-purpose tools:
1. `search_travel_packages(query: str)`: Searches package catalog by destination or theme.
2. `calculate_trip_budget(package_name: str, num_travelers: int, room_upgrade: bool, private_guide: bool)`: Computes base and upgraded package totals.
3. `lookup_policy_rule(topic: str)`: Checks verified corporate rules on flights, visas, cancellations, and bookings.
4. `record_customer_lead(name: str, email: str, notes: str)`: Records customer consultation inquiries into `logs/c4_leads.jsonl`.
5. `record_unanswered_inquiry(question: str)`: Logs gaps or unapproved questions into `logs/c4_feedback.jsonl`.

### Environment Setup (local Jupyter or Google Colab)
Run the next cell first.

**Locally:** it does nothing extra. Keys are read from `.env` in the project folder.

**In Google Colab:** the notebook needs the rest of this repository (`react_core.py`, `experiments.py`, `logs/`), not just this file. Either:
1. keep `REPO_URL` below (it clones this repository), **or**
2. upload the whole `WanderWise-C4-ReAct-Agent` folder to Google Drive at `MyDrive/WanderWise-C4-ReAct-Agent`; the cell will ask to mount Drive.

Then add your API key in Colab's **Secrets** panel (the key icon in the left sidebar), for example `OPENROUTER_API_KEY`, and switch on "Notebook access". Optionally add `LLM_PROVIDER` too. Key values are never printed.

In [ ]:
# --- Setup: works locally (Jupyter) and in Google Colab ---
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Jadjnm/WanderWise-C4-ReAct-Agent.git"  # Colab only; leave as is to clone this repository
DRIVE_DIR = Path("/content/drive/MyDrive/WanderWise-C4-ReAct-Agent")
SECRET_NAMES = ["LLM_PROVIDER", "OPENROUTER_API_KEY", "OPENROUTER_MODEL", "GEMINI_API_KEY", "GEMINI_MODEL",
                "OPENAI_API_KEY", "AWS_BEARER_TOKEN_BEDROCK", "AWS_REGION", "BEDROCK_MODEL_ID", "ENABLE_REFLECTION"]

if IN_COLAB:
    clone_dir = Path("/content/WanderWise-C4-ReAct-Agent")
    if REPO_URL and not clone_dir.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(clone_dir)], check=True)
    candidates = [clone_dir, DRIVE_DIR]
    if not any((c / "react_core.py").exists() for c in candidates):
        from google.colab import drive
        drive.mount("/content/drive")
    project = next((c for c in candidates if (c / "react_core.py").exists()), None)
    if project is None:
        raise FileNotFoundError(
            f"Project files not found. Set REPO_URL above, or upload the WanderWise_Agent_Project folder to {DRIVE_DIR}")
    os.chdir(project)

    # Install the project's requirements, except `notebook` (Colab ships its own Jupyter)
    reqs = [line.strip() for line in (project / "requirements.txt").read_text().splitlines()
            if line.strip() and not line.startswith("#") and not line.startswith("notebook")]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *reqs], check=True)

    # API keys from Colab Secrets (there is no .env file in Colab)
    from google.colab import userdata
    loaded = []
    for name in SECRET_NAMES:
        try:
            value = userdata.get(name)
        except Exception:  # secret not defined, or notebook access not granted
            value = None
        if value:
            os.environ[name] = value
            loaded.append(name)
    print("Secrets loaded:", ", ".join(loaded) or "none (add one in the Secrets panel)")

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print(f"Project folder: {Path.cwd().name} | Colab: {IN_COLAB} | Python {sys.version.split()[0]}")

### Imports
Loads the ReAct agent module and reports the active provider.

In [ ]:
import os, sys, json
from pathlib import Path
from dotenv import load_dotenv

BASE_DIR = Path.cwd()
load_dotenv(BASE_DIR / '.env')

import langgraph
import importlib.metadata
try:
    lg_ver = importlib.metadata.version('langgraph')
except Exception:
    lg_ver = '1.2.15'

import react_core as core

print('Python Version: ' + sys.version.split()[0])
print('LangGraph Version: ' + lg_ver)
print('Environment and dependencies initialized successfully.')

provider = core.get_active_provider()
print(f'Active provider: {provider} | model: {core.get_active_model(provider)}')
if provider == 'simulated':
    print('WARNING: no API key found -> offline simulator. Add a key to .env before running experiments.')


## 2. Tools & Business Reference
Here we inspect the tools and catalog definitions available to the ReAct agent.

In [ ]:
print("=== Available Tools ===")
for name in core.TOOLS_BY_NAME:
    print(f"• {name}")

# Direct tool execution test
search_res = core.search_travel_packages("bali")
calc_res = core.calculate_trip_budget("Bali Bliss", 2, room_upgrade=True)

print("\n--- Tool Output: calculate_trip_budget ---")
print(json.dumps(calc_res, indent=2))

## 3. Persona Design & LLM Configuration

Each persona couples prompt traits with a matching sampling configuration:

| Persona | Role & Tone | Temperature | Top-P | Top-K | Max tokens | Penalties | Description & Intended Use |
|---|---|---|---|---|---|---|---|
| **1. Friendly Advisor (Aria)** | Warm, welcoming, encouraging, empathetic, and conversational. | `0.70` | `0.90` | `40` | `600` | Freq: `0.1`, Pres: `0.2` | Ideal for first-time travelers, honeymooners, and general vacation exploration. Emphasizes excitement and personal care. |
| **2. Strict Auditor (Marcus)** | Analytical, precise, objective, concise, and uncompromising on guardrails. | `0.10` | `0.50` | `10` | `400` | Freq: `0.4`, Pres: `0.0` | Ideal for budget scrutiny, inclusion/exclusion audits, and strict policy enforcement. Never speculates. |
| **3. Creative Curator (Elena)** | Evocative, sensory, culturally immersed, and experiential. | `0.85` | `0.95` | `64` | `800` | Freq: `0.2`, Pres: `0.5` | Ideal for culture enthusiasts and off-the-beaten-path travelers seeking unique moments within package boundaries. |

In [ ]:
# Build per-setting parameter rationale table dynamically from PERSONA_PROFILES
from IPython.display import Markdown, display

rows = [
    "| Persona | Parameter | Setting | Rationale |",
    "|---|---|---|---|",
]
for pkey, p in core.PERSONA_PROFILES.items():
    for param, val in p["llm_config"].items():
        rat = p.get("config_rationale", {}).get(param, "")
        rows.append(f"| {p['name']} | `{param}` | `{val}` | {rat} |")

display(Markdown("\n".join(rows)))


## 4. Clear Instructions Design with Worked Example

Instruction design principles:
1. Keep each instruction focused on a single purpose.
2. Explicit inputs and outputs format.
3. At least one worked example inside the instructions.
4. No conflicts between persona and instructions.

In [ ]:
print("=== Dedicated Instructions & Worked Example ===")
print(core.DEDICATED_INSTRUCTIONS)

In [ ]:
# Persona / Instruction Conflict Audit
print("=== Persona / Instruction Conflict Audit ===")
conflicts = core.check_persona_conflicts()
for persona, phrase, rule, evidence in conflicts:
    print(f"[{persona}] '{phrase}'")
    print(f"  Conflict: {rule}")
    print(f"  Risk:     {evidence}\n")

print("Logged evidence correlation: all 4 airfare quotes came from Aria or Elena; Marcus quoted none.")


## 5. Manual ReAct Loop Implementation via LangGraph

The ReAct loop is implemented manually using LangGraph states and edges without prebuilt agent executors.

### State Graph Architecture
Our manual LangGraph state graph uses:
* **State (`ReActAgentState`):** Tracks the query, persona, prompt toggles (CoT, few-shot, instruction placement, instructions mode), loop guard toggles and events, resolved LLM config, provider/model, conversation history, the Thought/Action/Observation trace, the pending action, the final answer, format-error count, any LLM error, and the raw LLM calls.
* **Node 1: `reason`:** Builds the prompt (persona + instructions + tools, as a system prompt or inside the user message), calls the LLM with the persona's sampling configuration, and parses the reply into Thought and Action or Final Answer.
* **Node 2: `act`:** Dispatches the requested tool, executes it safely, and appends the Observation to the trace. Under `loop_guards=True`, if the exact same tool name and arguments already ran earlier in this turn, execution is blocked and an error observation directs the model to use its earlier observation or give its Final Answer.
* **Node 3: `force_answer`:** Triggered when the step limit (5 iterations) is reached without a Final Answer or LLM error under `loop_guards=True`. Makes ONE more LLM call prompting the agent to answer strictly from observations so far or state what is missing instead of guessing (`stopped_reason="forced_answer"`).
* **Conditional Edge (`should_continue`):**
  * Action requested -> `act` (then back to `reason`).
  * Reply had neither an Action nor a Final Answer -> back to `reason` with a format reminder.
  * 5-step limit reached -> `force_answer` if `loop_guards=True`, else `END` (returns canned "I won't guess" message).
  * Final Answer with `guardrails=True` -> `guard` -> `END`.
  * Final Answer with `guardrails=False` -> `END`.
  * LLM error -> `END`.
* **Node 4 (optional): `guard`:** deterministic code checks on the finished answer. If a policy lookup said the question *must be logged* and the model skipped `record_unanswered_inquiry`, the guard calls it and adds it to the trace; any sentence quoting an airfare dollar amount is removed and replaced with a short explanation. Guardrails are **off by default**, so every experiment in section 6 measures the model and prompt alone; the Gradio app turns them on.

**Action detection:** the parser accepts `Action: tool(arg="value")` (also markdown-wrapped, positional, JSON-style booleans, and parentheses inside quoted strings, via Python's AST). Anything the model writes after its own Action, such as an invented `Observation:`, is discarded, and a stop sequence on `\nObservation:` stops it from inventing one in the first place: only the `act` node may produce Observations.


In [ ]:
# Inspect compiled LangGraph workflow
graph = core.build_react_graph()
print("LangGraph Workflow compiled successfully:", graph)

def show(res):
    print(f"Persona: {res['persona']} | provider: {res['provider']}/{res['model']} | stopped: {res['stopped_reason']}")
    for s in res['steps']:
        print(f"[{s['type'].upper()}]: {s['content']}")
    print(f"\n--- Final Answer ---\n{res['final_answer']}\n")

# Demonstration test using Marcus (Strict Auditor)
query = "How much would a 7-day trip to Bali cost for 2 people with deluxe room upgrade?"
show(core.run_react_agent(query, persona_key="strict_auditor", use_cot=True))


### Multi-turn memory
The last few chat messages are passed back into the prompt as short-term memory, so a follow-up like *"yes, and add a private guide"* is resolved against the earlier turn.

In [ ]:
history = []
for msg in ["How much is Swiss Alpine Discovery for 2 people?", "Great, and with a private guide?"]:
    res = core.run_react_agent(msg, persona_key="friendly_advisor", history=history)
    print(f"TRAVELER: {msg}")
    show(res)
    history += [{"role": "user", "content": msg}, {"role": "assistant", "content": res["final_answer"]}]


### Code guardrails
The experiments showed two recurring model failures: skipping `record_unanswered_inquiry` after a policy lookup, and the warm personas quoting airfare ranges. The optional `guard` node enforces both rules in code. The cell below runs both questions with guardrails on; any intervention is listed under "Guardrail".

In [ ]:
for persona, q in [("friendly_advisor", "What happens if I need to cancel my Santorini trip because of a storm?"),
                   ("creative_curator", "Do you book flights to Rome, and roughly how much is the airfare?")]:
    res = core.run_react_agent(q, persona_key=persona, guardrails=True)
    print(f"TRAVELER: {q}")
    show(res)
    print("Guardrail:", res["guardrail_events"] or "no intervention needed", "\n" + "-" * 80)

## 6. Configuration Experimentation & Comparison

All experiments were run through `experiments.py`, which scores every run automatically (expected tool called, required facts present, hallucination patterns such as invented airfares or a wrong "grand total", steps, format errors, latency) and logs the full trace, raw model output, applied/ignored parameters and token usage.

**Experiment design (6 test cases per condition):**
| Suite | What varies | Held fixed |
|---|---|---|
| personas | Aria / Marcus / Elena (prompt traits + matching sampling config) | baseline prompt |
| prompting | CoT on/off, few-shot vs zero-shot, system prompt vs in-message instructions (one factor at a time) | Marcus |
| params | temperature, top-p, top-k, max tokens, frequency/presence penalty, second model | Aria |
| models | every persona on four extra free models | baseline prompt |

**Models:** `xingchen-agi/xing4.0-29b-a4b:free` (OpenRouter, supports all 7 parameters) and `claude-haiku-4.5` on AWS Bedrock ran the personas/prompting/params suites; four more free OpenRouter models ran the models suite. In total, 300 runs completed.

The cells below **load the logged results** (`logs/c4_experiments.jsonl`, `logs/c4_experiments_bedrock.jsonl`) and re-score them with the current scorer. Set `RUN_NEW = True` only to run the suites again (hundreds of LLM calls).

In [ ]:
import experiments as exp
from IPython.display import Markdown, display

RUN_NEW = False   # True re-runs the suites on the active provider (slow; hundreds of LLM calls)
SLEEP_S = 5       # pause between runs when RUN_NEW is True (free-tier rate limits)
LOGS = {"OpenRouter": "c4_experiments.jsonl", "Bedrock": "c4_experiments_bedrock.jsonl"}


def logged(suite=None, log=None):
    # Re-scored runs (latest attempt per run) from one log, or from all logs
    rows = []
    for name in ([log] if log else LOGS.values()):
        exp.EXPERIMENT_LOG = core.BASE_DIR / "logs" / name
        rows += exp.rescore(exp.load_records())
    exp.EXPERIMENT_LOG = core.BASE_DIR / "logs" / LOGS["OpenRouter"]
    return [r for r in rows if suite is None or r["suite"] == suite]


def show_suite(suite, **run_kwargs):
    if RUN_NEW:
        display(Markdown(exp.summarize(exp.run_suite(suite, sleep_s=SLEEP_S, **run_kwargs))))
        return
    for provider, log in LOGS.items():
        rows = logged(suite, log)
        if rows:
            display(Markdown(f"**{provider}**\n\n" + exp.summarize(rows)))


for c in exp.TEST_CASES:
    print(f"{c['id']:<30} expects {c['expected_tools']}")

### 6.1 Persona comparison
Same six cases, each persona with its own prompt traits and LLM configuration.

In [ ]:
show_suite("personas")

### 6.2 Prompt engineering: CoT, zero-shot vs few-shot, system prompt vs in-message instructions
One factor changed at a time from the baseline (CoT on, few-shot, instructions in the system prompt).

In [ ]:
show_suite("prompting", persona="strict_auditor")

### 6.3 Configuration sweep
Aria held fixed; hyperparameters varied one factor at a time against her defaults (temperature, top-p, top-k, max tokens, frequency/presence penalty, plus a second model). Parameters a model does not accept are recorded per run: xing4.0 applied all seven; Claude Haiku 4.5 on Bedrock ignores both penalties and refuses top-p together with temperature, and Bedrock caps temperature at 1.0 (the `temperature=1.2` condition ran at 1.0 and is recorded under `adjusted_params`).

**Reading these tables:** each condition has 6 runs, so one failure moves a pass rate by 17 points. On Haiku, the top-p and penalty conditions were sent with exactly the same effective settings as "persona defaults", yet scored between 83% and 100%: that spread is the noise floor.

In [ ]:
show_suite("params", persona="friendly_advisor")

### 6.4 Qualitative comparison
Same question, three personas, final answers side by side (useful for the tone discussion).

In [ ]:
q = "Tell me about the highlights of the Japan package."
for key in core.PERSONA_PROFILES:
    r = core.run_react_agent(q, persona_key=key)
    print(f"=== {r['persona']} (T={r['llm_config']['temperature']}) ===\n{r['final_answer']}\n")


### 6.5 Model comparison
The same baseline prompt and per-persona settings on every model: 3 personas x 6 cases = 18 runs per model (xing4.0 and Haiku come from the personas suite, which used identical settings).

In [ ]:
if RUN_NEW:
    print("Run: python experiments.py --suite models --models <ids> --sleep 5")
show_suite("models")

# One row per model on identical settings (personas + models suites), errors excluded
base = [r for r in logged() if r["suite"] in ("personas", "models") and not r["error"]]
lines = ["| Model | Pass | Loops | Skipped required tool | Wrong/invented numbers | Format errors | Avg words | Avg latency (s) |",
         "|---|---|---|---|---|---|---|---|"]
for model in sorted({r["model"] for r in base}, key=lambda m: -sum(x["metrics"]["passed"] for x in base if x["model"] == m)):
    rr = [r for r in base if r["model"] == model]
    loops = sum(r["stopped_reason"] == "max_iterations" for r in rr)
    skipped = sum(not r["metrics"]["tools_ok"] and r["stopped_reason"] != "max_iterations" for r in rr)
    wrong = sum(r["stopped_reason"] == "final_answer" and r["metrics"]["tools_ok"]
                and (bool(r["metrics"]["hallucination_hits"]) or not r["metrics"]["facts_ok"]) for r in rr)
    lines.append(f"| {model.split('/')[-1]} | {sum(r['metrics']['passed'] for r in rr)}/{len(rr)} | {loops} | {skipped} | {wrong} | "
                 f"{sum(r['format_errors'] for r in rr)} | {sum(r['metrics']['answer_words'] for r in rr) / len(rr):.0f} | "
                 f"{sum(r['latency_s'] for r in rr) / len(rr):.1f} |")
display(Markdown("\n".join(lines)))

### 6.6 Persona effect pooled over all six models

In [ ]:
lines = ["| Persona | Pass (all models) | Avg words | Airfare estimates on the flights question |", "|---|---|---|---|"]
everything = [r for r in logged() if not r["error"]]
for key, p in core.PERSONA_PROFILES.items():
    rr = [r for r in base if r["persona_key"] == key]
    flights = [r for r in everything if r["persona_key"] == key and r["case_id"] == "policy_flights" and r["stopped_reason"] == "final_answer"]
    quotes = sum(bool(r["metrics"]["hallucination_hits"]) for r in flights)
    lines.append(f"| {p['name']} | {sum(r['metrics']['passed'] for r in rr)}/{len(rr)} | "
                 f"{sum(r['metrics']['answer_words'] for r in rr) / len(rr):.0f} | {quotes}/{len(flights)} (all suites) |")
display(Markdown("\n".join(lines)))

### 6.7 Instructions ablation

Ablation testing the operational rules while keeping format contracts constant:
- **`full` (baseline):** Includes the complete `INSTRUCTION_RULES` (both input/output format and the 8 operational rules) plus worked few-shot examples.
- **`format_only`:** Keeps the `INPUT YOU RECEIVE` and `OUTPUT YOU MUST RETURN` format contracts (needed for parsing Thought/Action/Observation/Final Answer) and tool descriptions, but removes all operational rules (no pricing constraints, no policy inquiry requirements) and all worked examples.

Ran on the default model (`dots-studio/dots-3-note-preview:free`), comparing `full` vs `format_only` across all three personas.


In [ ]:
show_suite("instructions")


### 6.8 Loop guards

Evaluates loop prevention mechanisms on Marcus (`strict_auditor`) using repeated runs on the challenging policy cases (`policy_flights` and `policy_cancellation_unknown`):
- **Repeat-action block (`act` node):** If the exact same tool name and arguments are called more than once in the same turn, execution is intercepted, an error observation is returned, and a loop-guard event is recorded.
- **Forced final answer (`force_answer` node):** When reaching the 5-step limit without a final answer or LLM error, one final LLM call instructs the model to answer strictly from observations gathered so far rather than guessing (`stopped_reason="forced_answer"`).


In [ ]:
show_suite("loop_guards")


> ✏️ **Draft answers**, written from the logged results above. Rewrite them in your own words before submitting, and delete this note.

## 7. Reflection Questions (15 Points)

### Question 1: Which persona gave the most helpful or natural results?
**Answer:**
On **accuracy, the personas were effectively tied**. Pooled over six models on identical settings, Aria passed 33/36 runs, Marcus 31/36 and Elena 32/36 (section 6.6). The persona mainly changed **style**: Marcus answered in about 61 words with itemized figures, while Aria and Elena wrote about 97–99 words in a warmer, more descriptive voice.

For a customer-facing concierge, **Aria gave the most natural results**: her answers read like a helpful travel agent rather than an audit report. Each persona had a characteristic failure, though:
* **Aria and Elena** produced **all four invented airfare estimates** in our logs (e.g. "typical round-trip flights to Rome range from $600 to $1,200"), despite the rule never to invent flight prices. Marcus quoted none in 11 answers to the flights question.
* **Marcus** was the persona most likely to **over-verify**: on xing4.0 he kept re-checking policies until the 5-step limit, and on Claude Haiku he re-added the guide fee to the calculator's total ($5,100 instead of $4,700) in all three runs that used chain-of-thought plus few-shot examples.

So Aria is the most helpful default for general questions, and Marcus is the safer choice when the answer is a price or a policy.

### Question 2: Which prompt and configuration combination performed best for your use case?
**Answer:**
**The model mattered far more than the persona, the prompt style or the sampling parameters.** On identical settings (section 6.5): `dots-3-note-preview` 18/18, `nemotron-3.5-super-vl` 18/18, `nemotron-3-super-120b` 17/18, Claude Haiku 4.5 16/18, xing4.0 14/18, `laguna-s-2.1` 13/18. Three free models matched or beat the paid Haiku on this task, and ran in about 5–10 seconds per query.

**Prompting choices helped or hurt depending on the model:**
* On **xing4.0**, chain-of-thought plus few-shot examples was clearly better: 12/12 runs passed with both, against 12/18 when either was removed (one-sided Fisher p ≈ 0.03). Every failure without them was a loop on a policy question.
* On **Claude Haiku** it was the other way round: every condition that removed CoT or the examples scored 18/18, while the two with both scored 10/12. Both failures were the $5,100 over-calculation, so the extra scaffolding made the stronger model over-reason.
* **System prompt vs in-message instructions** made no difference on either model (6/6 vs 6/6 on xing4.0, 5/6 vs 5/6 on Haiku).

**Sampling parameters:** no single change had a measurable effect at 6 runs per condition. xing4.0's 60 params runs passed at 77% overall, and every condition was within one or two failures of that. The one clear effect was `max_tokens=120`, which cut answers from about 135 to 43 words on Haiku (78 to 52 on xing4.0) with no loss of correctness.

**Best combination for this use case:** a strong instruction-following model (`dots-3-note-preview` or `nemotron-3.5-super-vl`), CoT and few-shot examples in the system prompt, and Aria's persona and settings for customer-facing answers.

### Question 3: How well did your agent reason and use tools?
**Answer:**
Generally well. Across 300 completed runs on 8 models, the agent called the required tools in **280/300 (93%)**, took **2.8 steps** on average, and produced correctly formatted replies everywhere except on `laguna-s-2.1` (22 format errors). Typical traces are clean multi-step chains: `search_travel_packages` → `calculate_trip_budget` → Final Answer for prices, and `lookup_policy_rule` → `record_unanswered_inquiry` → Final Answer for questions with no approved policy.

The failures fell into three clear patterns:
1. **Loops.** The model kept re-checking until the 5-step limit (12 of xing4.0's 108 runs). The circuit breaker then returned an honest "I won't guess" message instead of an invented answer.
2. **Skipping a tool and answering from memory.** In 4 runs, xing4.0 told the traveler their question had been logged or passed to "our guest-experience team" without ever calling `record_unanswered_inquiry`. `laguna-s-2.1` invented Bali totals ($4,400 and $4,300) without calling the calculator.
3. **Not trusting the Observation.** In 7 runs, the calculator returned $4,700 for the Swiss trip, but the model re-added the $400 guide fee and reported $5,100.

Model size also showed clearly: a 2.6B model (`liquid/lfm-2.5-2.6b`) could not keep to the Thought/Action format at all, while the 120B+ models made no format errors.

### Question 4: Did your instructions block change the agent's behavior in a noticeable way? How?
**Answer:**
Yes. The clearest evidence comes from removing the worked examples from the instructions block (the zero-shot conditions):
* **On xing4.0 the examples were decisive.** With them, 16/18 runs passed; without them, 8/12. On the two policy questions it went from 4/6 to 0/4. The second example demonstrates exactly the "look up the policy → nothing approved → log the question" path, and without it the model looped instead of logging.
* **On Claude Haiku the examples were not needed** (12/12 without them). The rules alone were enough for the stronger model.
* **The examples shaped wording as much as actions.** Answers to the cancellation question echoed the example's phrasing ("we don't have published cancellation or refund terms yet, so I won't guess"). In 4 xing4.0 runs the model copied the example's claim that the question had been passed to "our guest-experience team" *without* calling the tool. It learned what to say before reliably learning what to do.
* **The explicit output format worked.** Specifying "one Thought line, then exactly one Action or Final Answer, never an Observation", together with the parser, gave zero format errors for 7 of the 8 models.
* **Where a rule competed with the persona, the persona sometimes won.** "Never invent flight prices" was followed in 36 of 40 flights answers, and all four exceptions came from the warm personas, who framed a price range as being helpful.

Limitation: we removed the examples, but not the rules themselves, so the effect of the rules alone was not measured separately.

### Question 5: What were the biggest challenges in implementation?
**Answer:**
1. **Unreliable providers.** The OpenAI key ran out of credits, `gemini-2.5-flash` was retired for new accounts, and its replacement allowed only 20 free requests per day. Free OpenRouter models were often overloaded (429/502 errors) or blocked for this kind of app. Our first notebook run silently fell back to a canned simulator, which produced meaningless results. We changed the design so that a failed call is always reported as an error, added retries for per-minute limits and fail-fast for daily quotas, and added `--resume` to re-run only failed experiments.
2. **Every provider handles parameters differently.** Gemini 3.8 rejects penalties; Claude rejects `top_p` together with `temperature` and any temperature above 1.0; OpenRouter silently drops parameters a model does not support. We record the applied, ignored and adjusted parameters on every run, so comparisons stay honest.
3. **Parsing free-form ReAct output.** Models wrapped actions in markdown, put parentheses inside arguments, invented their own `Observation:` lines, and some "reasoning" models leaked hidden reasoning into the reply. We addressed this with an AST-based argument parser, stop sequences, a format-retry edge in the graph, and disabling hidden reasoning where the provider allows it.
4. **Evaluating fairly.** The first automatic scorer marked "No group tour is planned" as a hallucination and missed a wrong "Grand Total" that followed a correct breakdown. We made it negation-aware, re-scored every logged run, and checked each changed verdict by hand. With 6 runs per condition, we also had to treat one-run differences as noise and pool conditions before drawing conclusions.

## 8. Interactive Gradio Demonstration
Launch the interactive web application to demo persona switching and inspect the live ReAct trace.

In [ ]:
from c4_app import build_c4_demo
demo = build_c4_demo()
demo.launch(share=False)